# Config

In [10]:
import os
import importlib.metadata
from agents import Agent, Runner

In [11]:
assert os.getenv("OPENAI_API_KEY"),("OPENAI_API_KEY is missing inside the Jupyter container.")

In [3]:
MODEL = "gpt-6-luna"

In [13]:
print("OpenAI Agents SDK:", importlib.metadata.version("openai-agents"))
print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Model:", MODEL)

OpenAI Agents SDK: 0.22.3
API key loaded: True
Model: gpt-6-luna


<hr>

# Smoke test

In [14]:
agent = Agent(
    name="Smoke Test Agent",
    instructions="Reply with exactly: OPENAI_AGENTS_SDK_OK",
    model=MODEL,
)

result = await Runner.run(agent, "Run the smoke test.")

print("Agent response:", result.final_output)
print("Usage:", result.context_wrapper.usage)

Agent response: OPENAI_AGENTS_SDK_OK
Usage: Usage(requests=1, input_tokens=26, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=40, output_tokens_details=OutputTokensDetails(reasoning_tokens=27), total_tokens=66, request_usage_entries=[RequestUsage(input_tokens=26, output_tokens=40, total_tokens=66, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens_details=OutputTokensDetails(reasoning_tokens=27))])


<hr>

# Web search

In [15]:
from agents import Agent, Runner, WebSearchTool

In [18]:
research_agent = Agent(
    name = "Web Search Demo",
    model = MODEL,
    instructions = (
        "Use web search. Answer with two factual findings."
        "For each finding, include the source title and URL. "
        "If you cannot verify a fact from a source, say so."
    ),
    tools=[WebSearchTool(search_context_size="low")],
)

In [22]:
result = await Runner.run(
    research_agent,
    "Find official OpenAI Agents SDK documentation explaining how web search"
    "and session history work. Give me the documentation titles and URLs"
)
print(result)

RunResult:
- Last agent: Agent(name="Web Search Demo", ...)
- Final output (str):
    1. **Web search — “Tools”**  
       The Agents SDK documents `WebSearchTool` as a hosted tool that lets an agent search the web.  
       URL: https://openai.github.io/openai-agents-python/tools/ ([openai.github.io](https://openai.github.io/openai-agents-python/tools/?trk=article-ssr-frontend-pulse_little-text-block&utm_source=openai))
    
    2. **Session history — “Sessions”**  
       The SDK retrieves a session’s conversation history before each run and stores new items after the run, so later turns can use the earlier context.  
       URL: https://openai.github.io/openai-agents-python/sessions/ ([openai.github.io](https://openai.github.io/openai-agents-python/sessions/?utm_source=openai))
- 3 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [26]:
print(result.final_output)
print(f"Uageg: {result.context_wrapper.usage}")

1. **Web search — “Tools”**  
   The Agents SDK documents `WebSearchTool` as a hosted tool that lets an agent search the web.  
   URL: https://openai.github.io/openai-agents-python/tools/ ([openai.github.io](https://openai.github.io/openai-agents-python/tools/?trk=article-ssr-frontend-pulse_little-text-block&utm_source=openai))

2. **Session history — “Sessions”**  
   The SDK retrieves a session’s conversation history before each run and stores new items after the run, so later turns can use the earlier context.  
   URL: https://openai.github.io/openai-agents-python/sessions/ ([openai.github.io](https://openai.github.io/openai-agents-python/sessions/?utm_source=openai))
Uageg: Usage(requests=1, input_tokens=12748, input_tokens_details=InputTokensDetails(cache_write_tokens=4434, cached_tokens=0), output_tokens=327, output_tokens_details=OutputTokensDetails(reasoning_tokens=184), total_tokens=13075, request_usage_entries=[RequestUsage(input_tokens=12748, output_tokens=327, total_token

In [27]:
for index, item in enumerate(result.new_items, start=1):
    print(index, type(item).__name__)

1 ToolCallItem
2 ReasoningItem
3 MessageOutputItem


<hr>

# ทดสอบการดึงไฟล์ที่เป็น Benchmark

In [28]:
import json
from pathlib import Path

path = Path("shared/benchmark_cases.json")
print("File:", path.resolve())

with path.open(encoding="utf-8") as file:
    cases = json.load(file)

print(cases.keys())

File: /workspace/shared/benchmark_cases.json
dict_keys(['long_prompt_math'])


In [31]:
import json
import time
from agents import Agent, Runner

In [32]:
with open("shared/benchmark_cases.json", encoding="utf-8") as file:
    cases = json.load(file)
cases = cases["long_prompt_math"]
cases

{'prompt': 'You are reviewing a fictional company. Use only the figures below; do not use web search or add outside facts. FY2024: revenue 1,000 million THB, net income 80 million THB, total debt 300 million THB, cash 100 million THB. FY2025: revenue 1,200 million THB, net income 102 million THB, total debt 280 million THB, cash is not reported. Calculate revenue growth from FY2024 to FY2025, net income growth, net margin for both years, and net debt for each year only when both debt and cash are available. Show the formula for every calculation. Treat missing cash as unavailable, never as zero. Return an executive summary, a financial table, calculation notes, and data limitations. Label all figures as supplied test data, not externally verified facts.',
 'requirements': ['revenue_growth_is_20_percent',
  'net_income_growth_is_27_5_percent',
  'fy2024_net_margin_is_8_percent',
  'fy2025_net_margin_is_8_5_percent',
  'fy2024_net_debt_is_200_million_thb',
  'fy2025_net_debt_is_unavailab

In [81]:
from datetime import datetime, timezone
import time

run_id = datetime.now(timezone.utc).strftime(
    "openai-long-math-%Y%m%dT%H%M%SZ"
)

started_at = time.perf_counter()
result = await Runner.run(math_agent, case["prompt"])
latency_s = time.perf_counter() - started_at

print("Run ID:", run_id)
print("Answer:", result.final_output)
print("Latency (seconds):", round(latency_s, 2))
print("Usage:", result.context_wrapper.usage)

Run ID: openai-long-math-20260929T080352Z
Answer: ## Executive summary

Using only the supplied test data, revenue grew **20%** and net income grew **27.5%** from FY2024 to FY2025. Net margin increased from **8.0%** to **8.5%**. FY2024 net debt was **200 million THB**. FY2025 net debt is **unavailable** because cash was not reported.

All figures below are **supplied test data, not externally verified facts**.

## Financial table

| Metric | FY2024 | FY2025 |
|---|---:|---:|
| Revenue (million THB) | 1,000 | 1,200 |
| Net income (million THB) | 80 | 102 |
| Total debt (million THB) | 300 | 280 |
| Cash (million THB) | 100 | Not reported |
| Revenue growth from FY2024 | — | 20% |
| Net income growth from FY2024 | — | 27.5% |
| Net margin | 8.0% | 8.5% |
| Net debt (million THB) | 200 | Unavailable |

## Calculation notes

- **Revenue growth, FY2024 to FY2025:**  
  \((\text{FY2025 revenue} - \text{FY2024 revenue}) \div \text{FY2024 revenue} \times 100\)  
  \(= (1{,}200 - 1{,}000) \div 

In [82]:
math_agent = Agent(
    name = "Long Prompt Math Test",
    model = MODEL,
    instructions = "Follow every requirement in the user request. Do not use tools.",
)

In [83]:
started_at = time.perf_counter()
result = await Runner.run(math_agent, case["prompt"])
latency_s = time.perf_counter() - started_at

In [84]:
print(result.final_output)

## Executive summary

Using only the **supplied test data (not externally verified facts)**, revenue increased **20%** and net income increased **27.5%** from FY2024 to FY2025. Net margin rose from **8.0%** to **8.5%**. FY2024 net debt was **200 million THB**. FY2025 net debt is **unavailable** because cash was not reported.

## Financial table

All reported amounts below are **supplied test data, not externally verified facts**. Amounts are in million THB.

| Metric | FY2024 | FY2025 |
|---|---:|---:|
| Revenue | 1,000 | 1,200 |
| Revenue growth vs. prior year | — | 20% |
| Net income | 80 | 102 |
| Net income growth vs. prior year | — | 27.5% |
| Net margin | 8.0% | 8.5% |
| Total debt | 300 | 280 |
| Cash | 100 | Unavailable |
| Net debt | 200 | Unavailable |

## Calculation notes

Calculations use the supplied test data:

- **Revenue growth, FY2024 to FY2025**  
  Formula: \((\text{FY2025 revenue} - \text{FY2024 revenue}) \div \text{FY2024 revenue} \times 100\)  
  Calculation: \((

In [85]:
print("Model:", MODEL)
print(f"Latency (sec): {round(latency_s, 2)}")
print(f"Usage: {result.context_wrapper.usage}")

Model: gpt-6-luna
Latency (sec): 7.45
Usage: Usage(requests=1, input_tokens=199, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=833, output_tokens_details=OutputTokensDetails(reasoning_tokens=157), total_tokens=1032, request_usage_entries=[RequestUsage(input_tokens=199, output_tokens=833, total_tokens=1032, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens_details=OutputTokensDetails(reasoning_tokens=157))])


<hr>

# Record result

In [86]:
import json
from datetime import datetime, timezone
from pathlib import Path

In [87]:
requirement_checks = {
    "revenue_growth_is_20_percent": True,
    "net_income_growth_is_27_5_percent": True,
    "fy2024_net_margin_is_8_percent": True,
    "fy2025_net_margin_is_8_5_percent": True,
    "fy2024_net_debt_is_200_million_thb": True,
    "fy2025_net_debt_is_unavailable": True,
    "missing_cash_is_not_treated_as_zero": True,
    "formulas_are_shown": True,
    "all_requested_sections_are_present": True,
    "data_is_labeled_as_supplied_test_data": True,
}

In [88]:
usage = result.context_wrapper.usage
requirements_total = len(requirement_checks)
requirements_met = sum(requirement_checks.values())

In [89]:
record = {
    "sdk": "openai-agents",
    "case": "long_prompt_math",
    "run_id": datetime.now(timezone.utc).strftime("openai-long-math-%Y%m%dT%H%M%SZ"),
    "model": MODEL,
    "latency_s": round(latency_s, 2),
    "usage": {
        "requests": usage.requests,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "reasoning_tokens": usage.output_tokens_details.reasoning_tokens,
        "total_tokens": usage.total_tokens,
    },
    "requirements_met": requirements_met,
    "requirements_total": requirements_total,
    "requirement_checks": requirement_checks,
    "run_item_types": [type(item).__name__ for item in result.new_items],
    "answer": result.final_output,
}

results_dir = Path("shared/results")
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / f"{record['run_id']}.json"
output_path.write_text(
    json.dumps(record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Checklist score: {requirements_met}/{requirements_total}")
print(f"Saved result to: {output_path.resolve()}")

Checklist score: 10/10
Saved result to: /workspace/shared/results/openai-long-math-20260929T080409Z.json


<hr>

In [90]:
import json
import statistics
from pathlib import Path

result_files = sorted(
    Path("shared/results").glob("openai-long-math-*.json")
)

records = [
    json.loads(path.read_text(encoding="utf-8"))
    for path in result_files
]

if len(records) != 3:
    print(f"Expected 3 results, found {len(records)}")
else:
    for record in records:
        usage = record["usage"]
        print(
            record["run_id"],
            f"| latency={record['latency_s']}s",
            f"| tokens={usage['total_tokens']}",
            f"| score={record['requirements_met']}/{record['requirements_total']}",
        )

    latencies = [r["latency_s"] for r in records]
    print("Median latency:", statistics.median(latencies), "seconds")
    print("Latency range:", min(latencies), "to", max(latencies), "seconds")

openai-long-math-20260929T080244Z | latency=7.22s | tokens=984 | score=10/10
openai-long-math-20260929T080319Z | latency=7.88s | tokens=1046 | score=10/10
openai-long-math-20260929T080409Z | latency=7.45s | tokens=1032 | score=10/10
Median latency: 7.45 seconds
Latency range: 7.22 to 7.88 seconds


<hr>

In [91]:
import json
from pathlib import Path

path = Path("shared/benchmark_cases.json")

with path.open(encoding="utf-8") as file:
    benchmark_cases = json.load(file)

benchmark_cases["web_research_ptt"] = {
    "prompt": (
        "Research PTT Public Company Limited (SET ticker: PTT) as of "
        "2026-09-29. Use the provided search and retrieval tools. "
        "Prioritize official PTT investor-relations materials, annual reports, "
        "financial statements, and SET disclosures. Verify the company's identity. "
        "Identify its most recently published completed fiscal year as of the "
        "cutoff date. Report revenue and net profit with amount, currency, period, "
        "and a source URL. Also report at least two material company developments "
        "published in the 12 months before the cutoff date, with their dates and "
        "source URLs. Cite a source URL for every factual claim. Label figures as "
        "reported or calculated. Explain missing information or conflicting sources "
        "instead of guessing. Do not give a buy or sell recommendation."
    ),
    "requirements": [
        "company_identity_verified",
        "cutoff_date_used",
        "latest_completed_fiscal_year_identified",
        "revenue_has_period_amount_currency_and_source",
        "net_profit_has_period_amount_currency_and_source",
        "at_least_two_recent_developments_with_dates",
        "factual_claims_have_source_urls",
        "official_sources_prioritized",
        "reported_and_calculated_figures_distinguished",
        "missing_or_conflicting_data_handled",
        "no_buy_or_sell_recommendation"
    ]
}

path.write_text(
    json.dumps(benchmark_cases, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Cases:", list(benchmark_cases.keys()))
print("Saved:", path.resolve())

Cases: ['long_prompt_math', 'web_research_ptt']
Saved: /workspace/shared/benchmark_cases.json


<hr>

In [1]:
import os
from agents.mcp import MCPServerStreamableHttp

assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is missing in the container."

async with MCPServerStreamableHttp(
    name="Tavily MCP",
    params={
        "url": "https://mcp.tavily.com/mcp/",
        "headers": {
            "Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}"
        },
        "timeout": 30,
    },
) as tavily_server:
    tools = await tavily_server.list_tools()
    print("Connected tools:", [tool.name for tool in tools])

Connected tools: ['tavily_search', 'tavily_extract', 'tavily_crawl', 'tavily_map', 'tavily_research']


In [4]:
from agents import Agent, Runner
from agents.model_settings import ModelSettings
from agents.mcp import MCPServerStreamableHttp

async with MCPServerStreamableHttp(
    name="Tavily MCP",
    params={
        "url": "https://mcp.tavily.com/mcp/",
        "headers": {
            "Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}"
        },
        "timeout": 30,
    },
) as tavily_server:

    search_agent = Agent(
        name="Tavily MCP Search Test",
        model=MODEL,
        instructions=(
            "Use the Tavily MCP search tool. "
            "Return three search results with their titles and URLs."
        ),
        mcp_servers=[tavily_server],
        model_settings=ModelSettings(tool_choice="required"),
    )

    result = await Runner.run(
        search_agent,
        "Search the web for the official PTT Public Company Limited "
        "investor relations website. Return the result titles and URLs.",
    )

print(result.final_output)
print("Run item types:", [type(item).__name__ for item in result.new_items])
print("Usage:", result.context_wrapper.usage)

- [PTT Public Company Limited (Investor Relations)](https://investor.pttplc.com/en/ir-home)
- [IR Contact | PTT Public Company Limited](https://investor.pttplc.com/en/information-request/ir-contact)
- [Presentation & Webcast | PTT Public Company Limited](https://investor.pttplc.com/en/downloads/presentations-webcasts)
Run item types: ['ToolCallItem', 'ToolCallOutputItem', 'MessageOutputItem']
Usage: Usage(requests=2, input_tokens=4361, input_tokens_details=InputTokensDetails(cache_write_tokens=2737, cached_tokens=1495), output_tokens=124, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=4485, request_usage_entries=[RequestUsage(input_tokens=1498, output_tokens=38, total_tokens=1536, input_tokens_details=InputTokensDetails(cache_write_tokens=1372, cached_tokens=0), output_tokens_details=OutputTokensDetails(reasoning_tokens=0)), RequestUsage(input_tokens=2863, output_tokens=86, total_tokens=2949, input_tokens_details=InputTokensDetails(cache_write_tokens=1365, 

<hr>

# Benchmark

In [8]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

from agents import Agent, Runner
from agents.mcp import (
    MCPServerStreamableHttp,
    create_static_tool_filter,
)

# โหลด benchmark case
with open("shared/benchmark_cases.json", encoding="utf-8") as file:
    cases = json.load(file)

case = cases["web_research_ptt"]

# สร้าง run ID ที่ไม่ซ้ำกัน แม้รันติดกันภายในวินาทีเดียว
now_utc = datetime.now(timezone.utc)
run_id = (
    now_utc.strftime("openai-ptt-research-%Y%m%dT%H%M%S")
    + f"{now_utc.microsecond:06d}Z"
)

# เริ่มจับเวลา
started_at = time.perf_counter()

# เชื่อม MCP แล้วรันโจทย์
async with MCPServerStreamableHttp(
    name="Tavily MCP",
    params={
        "url": "https://mcp.tavily.com/mcp/",
        "headers": {
            "Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}",
            "DEFAULT_PARAMETERS": json.dumps({
                "max_results": 3,
                "search_depth": "basic",
            }),
        },
        "timeout": 60,
    },
    tool_filter=create_static_tool_filter(
        allowed_tool_names=[
            "tavily_search",
            "tavily_extract",
        ]
    ),
) as tavily_server:

    research_agent = Agent(
        name="PTT Web Research Benchmark",
        model=MODEL,
        instructions=(
            "Use Tavily search and extract tools efficiently. "
            "Use at most two searches: one for FY2025 financial results "
            "and one for material company developments during the requested "
            "12-month period. Extract only the official PTT or SET pages "
            "needed to verify the claims. Stop when the requested facts have "
            "adequate sources. If dates or figures conflict, explain the "
            "conflict and do not guess. Include a source URL for every "
            "factual claim."
        ),
        mcp_servers=[tavily_server],
    )

    result = await Runner.run(
        research_agent,
        case["prompt"],
        max_turns=10,
    )

latency_s = time.perf_counter() - started_at

# สรุป usage
usage = result.context_wrapper.usage

# เก็บจำนวน tool calls และข้อความ error จาก MCP เพื่อช่วยตรวจรอบถัดไป
tool_call_count = sum(
    type(item).__name__ == "ToolCallItem"
    for item in result.new_items
)

tool_error_messages = []

for item in result.new_items:
    if type(item).__name__ == "ToolCallOutputItem":
        output = str(getattr(item, "output", ""))
        output_lower = output.lower()

        if "error" in output_lower or "timed out" in output_lower:
            tool_error_messages.append(output[:500])

# บันทึกผลดิบ โดยยังไม่ใส่คะแนน checklist
record = {
    "sdk": "openai-agents",
    "case": "web_research_ptt",
    "run_id": run_id,
    "status": (
        "completed_with_tool_error"
        if tool_error_messages
        else "completed"
    ),
    "model": MODEL,
    "latency_s": round(latency_s, 2),
    "usage": {
        "requests": usage.requests,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "reasoning_tokens": usage.output_tokens_details.reasoning_tokens,
        "total_tokens": usage.total_tokens,
    },
    "tool_call_count": tool_call_count,
    "tool_error_count": len(tool_error_messages),
    "tool_errors": tool_error_messages,
    "run_item_types": [type(item).__name__ for item in result.new_items],
    "requirements": case["requirements"],
    "answer": result.final_output,
}

# บันทึกไฟล์ผลลัพธ์
results_dir = Path("shared/results")
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / f"{run_id}.json"
output_path.write_text(
    json.dumps(record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# แสดงผล
print(result.final_output)
print("Run ID:", run_id)
print("Latency (seconds):", round(latency_s, 2))
print("Model requests:", usage.requests)
print("Tool calls:", tool_call_count)
print("Tool errors:", len(tool_error_messages))
print("Saved:", output_path)

MCP tool returned an error.


## PTT Public Company Limited — as of 29 September 2026

**Identity.** The company identifies itself as PTT Public Company Limited and its investor-relations site identifies its SET ticker as **PTT**. [PTT Investor Relations](https://investor.pttplc.com/en/ir-home)

**Latest completed fiscal year.** PTT’s latest completed fiscal year was **FY2025**: its investor-relations materials list 2025 annual reporting and financial statements, while the newer 2026 materials concern interim periods. [PTT Investor Relations](https://investor.pttplc.com/en/ir-home) [Financial Statements & MD&A](https://investor.pttplc.com/en/downloads/financial-statements-mda)

| FY2025 measure | Amount | Basis |
|---|---:|---|
| Revenue (sales and service revenue) | **THB 2,662,145 million** | **Reported** |
| Net profit attributable to equity holders of the company | **THB 90,166 million** | **Reported** |

Source for both figures: [PTT Financial Highlights](https://investor.pttplc.com/en/financial-information/fi

In [10]:
import json
from pathlib import Path

path = Path(
    "shared/results/openai-ptt-research-20260929T090028257774Z.json"
)

with path.open(encoding="utf-8") as file:
    saved = json.load(file)

print("Status:", saved.get("status"))
print("Tool errors:", json.dumps(saved.get("tool_errors", []), ensure_ascii=False, indent=2))
print("Usage:", json.dumps(saved.get("usage", {}), ensure_ascii=False, indent=2))

Status: completed_with_tool_error
Tool errors: [
  "An error occurred while running the tool. Please try again. Error: Request 'tools/call' timed out"
]
Usage: {
  "requests": 5,
  "input_tokens": 23104,
  "output_tokens": 2420,
  "reasoning_tokens": 1528,
  "total_tokens": 25524
}
